[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/069.%20The%20Shortest%20Path%20Problem/P69-Tier-1_executed.ipynb)

---


# 69. The Shortest Path Problem

## Tier 1: The Pen-and-Paper Method (Mathematical Formulation)

### Purpose and scope

This notebook formulates a shortest-path problem on a directed freight network as a binary linear optimization model. Its inputs are nodes, directed edges, an edge cost for each connection, and a source and target. Its outputs are a minimum-cost route, the selected edges, and checks of the flow constraints. A network plot and a small cost-sensitivity experiment help interpret the solution.

The model minimizes **edge cost**. The distance values stored with the edges are descriptive data and are not included in the objective. Node coordinates are used only for plotting. Costs are assumed to be non-negative. The implementation assumes distinct source and target nodes and at most one directed edge for each ordered pair of nodes. The network may contain multiple feasible source-to-target paths; the model selects a minimum-cost one. This example does not model capacities, travel times, uncertainty, or other operational constraints.

### Background: directed networks and paths

A directed graph is a pair $G=(V,E)$, where $V$ is a set of nodes and $E$ is a set of directed edges. An edge $(i,j)$ permits movement from node $i$ to node $j$; it does not imply that movement in the reverse direction is available. A path from source $s$ to target $t$ is a sequence of directed edges that joins $s$ to $t$. Its cost is the sum of the costs of its edges.

The example represents distribution centers by nodes and transportation links by edges. It seeks a minimum-cost route from node A (Los Angeles) to node G (New York). The edge costs, rather than the plotted coordinates or the separately recorded distances, determine which route is optimal.

### Mathematical formulation

The decision is made edge by edge: for each directed edge $(i,j)\in E$, the binary variable $x_{ij}$ is 1 if the edge is selected and 0 otherwise. Flow conservation connects these local decisions into a source-to-target flow. The model is:

$$
\begin{aligned}
\min\quad & \sum_{(i,j)\in E} c_{ij}x_{ij} \\
\text{subject to}\quad
 & \sum_{j:(i,j)\in E}x_{ij}-\sum_{j:(j,i)\in E}x_{ji}=b_i,
 && i\in V,\\
 & x_{ij}\in\{0,1\},
 && (i,j)\in E,
\end{aligned}
$$

Here, $c_{ij}$ is the cost of edge $(i,j)$, and $b_i=1$ for the source, $b_i=-1$ for the target, and $b_i=0$ for every other node. The left side of the flow constraint is net outflow: selected outgoing flow minus selected incoming flow. In the code, the binary variables are `edge_vars`, the edge costs are `edge.cost`, and the flow-balance constraints are added in `solve_shortest_path_mip`. The objective value is the sum of the costs of selected edges.

The source constraint sends one unit of flow out, the target constraint brings one unit in, and each intermediate node has balanced incoming and outgoing flow. The binary restrictions mean that each edge is either selected or not selected; they do not represent a continuous fraction of an edge. With strictly positive edge costs, a selected cycle would increase the objective and so cannot be part of an optimal solution. With non-negative costs that include zero-cost edges, an optimal solution can contain a zero-cost cycle without changing its objective. The supplied example has strictly positive edge costs. A solver reporting an optimal status certifies optimality for the formulated model. If multiple optimal routes exist, a solver may return any one of them.

### Solution sequence

1. Define the directed network and identify its source and target.
2. Create one binary decision variable for each edge.
3. Minimize the total cost of selected edges.
4. Add one flow-conservation constraint per node.
5. Solve the mixed-integer linear program, reconstruct the route, and verify the reported flow and cost.

The notebook uses PuLP with the CBC solver when PuLP is available. Its included fallback enumerates simple paths and is suitable only for small networks: a directed graph can have exponentially many simple paths as its number of nodes grows. Both routes retain the same objective and source-to-target shortest-path goal.

In [ ]:
from typing import Dict, List, Set, Tuple
from dataclasses import dataclass
import math

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# PuLP provides the mixed-integer optimization model used in this tier.
# The included path enumeration remains available for small instances when PuLP is absent.
try:
    import pulp
    PULP_AVAILABLE = True
    print("PuLP is available; the mixed-integer programming model will be used.")
except ImportError:
    PULP_AVAILABLE = False
    print("PuLP is not available; the small-network path enumeration fallback will be used.")

print("Libraries imported successfully.")

In [ ]:
@dataclass
class NetworkNode:
    """Represent a node in the freight network."""

    id: str
    name: str
    x: float  # Horizontal plotting coordinate; it does not affect route cost.
    y: float  # Vertical plotting coordinate; it does not affect route cost.


@dataclass
class NetworkEdge:
    """Represent a directed transportation link and its attributes."""

    from_node: str
    to_node: str
    cost: float
    distance: float


@dataclass
class ShortestPathProblem:
    """Define a shortest-path instance on a directed network."""

    nodes: Dict[str, NetworkNode]
    edges: List[NetworkEdge]
    source: str
    target: str


print("Network data structures are ready.")

In [ ]:
def create_freight_network() -> ShortestPathProblem:
    """Create the seven-node freight network used in this example."""

    nodes = {
        "A": NetworkNode("A", "Los Angeles", 0, 0),
        "B": NetworkNode("B", "Phoenix", 2, 1),
        "C": NetworkNode("C", "Denver", 1, 2),
        "D": NetworkNode("D", "Dallas", 3, 1),
        "E": NetworkNode("E", "Chicago", 4, 2),
        "F": NetworkNode("F", "Atlanta", 5, 1),
        "G": NetworkNode("G", "New York", 7, 2),
    }

    # Costs define the optimization objective; distances are recorded separately.
    edges = [
        NetworkEdge("A", "B", 2, 100),
        NetworkEdge("A", "C", 3, 150),
        NetworkEdge("B", "D", 4, 120),
        NetworkEdge("B", "C", 1, 80),
        NetworkEdge("C", "E", 6, 200),
        NetworkEdge("C", "D", 2, 90),
        NetworkEdge("D", "E", 3, 110),
        NetworkEdge("D", "F", 5, 180),
        NetworkEdge("E", "F", 2, 70),
        NetworkEdge("E", "G", 4, 160),
        NetworkEdge("F", "G", 4, 140),
    ]

    return ShortestPathProblem(nodes, edges, "A", "G")


problem = create_freight_network()
print(f"Network created with {len(problem.nodes)} nodes and {len(problem.edges)} directed edges.")
print(f"Source: {problem.source} ({problem.nodes[problem.source].name})")
print(f"Target: {problem.target} ({problem.nodes[problem.target].name})")

### Modeling the edge choices

The model creates one binary variable for every edge, not for every possible route. This representation lets the flow constraints connect local edge decisions into a source-to-target solution. The objective sums `edge.cost * edge_var` over all edges. At an intermediate node, the number of selected outgoing edges minus the number of selected incoming edges must be zero; at the source and target, the required net flows are 1 and -1, respectively.

The solver status matters: a route and objective value should be interpreted as an optimum only when the solver reports `Optimal`. The code also reconstructs a path from the selected edges and checks that a source-to-target route can be followed.

### How the small-network fallback enumerates paths

When PuLP is unavailable, `solve_shortest_path_enumeration` follows the same source-to-target objective by listing simple paths and comparing their costs. A simple path does not visit the same node twice. The recursive search keeps track of its current node and the nodes already visited, which prevents cycles from creating an endless search.

Let $P(v,S)$ denote the set of simple paths from the current node $v$ to target $t$ that do not revisit any node in the visited set $S$. Its base case and recursive transition are:

$$
P(v,S)=
\begin{cases}
\{[t]\}, & v=t,\\
\displaystyle\bigcup_{(v,w)\in E,\,w\notin S}\{[v]\mathbin{+}p:p\in P(w,S\cup\{w\})\}, & v\ne t.
\end{cases}
$$

Here, $v$ is the current node, $S$ is the set of nodes already visited, and $[v]\mathbin{+}p$ means prefixing node $v$ to path $p. In the code, `find_all_paths` implements this recursive search using `current`, `visited`, and `path`; `all_paths` collects completed paths. Each recursive call adds a node to the current path, and the function removes it again when that branch has been explored so that other branches can be considered. The number of simple paths can grow exponentially with the network size, so this fallback is intended for small networks. The primary method in this tier remains the binary flow optimization model.

In [ ]:
def solve_shortest_path_mip(problem: ShortestPathProblem) -> Dict:
    """Solve the binary flow model for a shortest path."""

    if not PULP_AVAILABLE:
        return solve_shortest_path_enumeration(problem)

    if problem.source not in problem.nodes or problem.target not in problem.nodes:
        raise ValueError("The source and target must both be nodes in the network.")
    if problem.source == problem.target:
        raise ValueError("The source and target must be distinct nodes.")
    if any(edge.cost < 0 for edge in problem.edges):
        raise ValueError("This notebook assumes non-negative edge costs.")
    if any(
        edge.from_node not in problem.nodes or edge.to_node not in problem.nodes
        for edge in problem.edges
    ):
        raise ValueError("Every edge endpoint must be a node in the network.")
    edge_pairs = [(edge.from_node, edge.to_node) for edge in problem.edges]
    if len(edge_pairs) != len(set(edge_pairs)):
        raise ValueError("Each ordered node pair may have at most one directed edge.")

    model = pulp.LpProblem("Shortest_Path", pulp.LpMinimize)

    # Binary variables encode each edge choice while retaining its 0-or-1 bounds.
    edge_vars = {}
    for edge in problem.edges:
        var_name = f"x_{edge.from_node}_{edge.to_node}"
        edge_vars[(edge.from_node, edge.to_node)] = pulp.LpVariable(
            var_name,
            lowBound=0,
            upBound=1,
            cat=pulp.LpBinary,
        )

    model += pulp.lpSum(
        edge.cost * edge_vars[(edge.from_node, edge.to_node)]
        for edge in problem.edges
    )

    # Net outflow is 1 at the source, -1 at the target, and 0 elsewhere.
    for node_id in problem.nodes:
        outgoing = pulp.lpSum(
            edge_vars[(edge.from_node, edge.to_node)]
            for edge in problem.edges
            if edge.from_node == node_id
        )
        incoming = pulp.lpSum(
            edge_vars[(edge.from_node, edge.to_node)]
            for edge in problem.edges
            if edge.to_node == node_id
        )

        if node_id == problem.source:
            model += outgoing - incoming == 1
        elif node_id == problem.target:
            model += outgoing - incoming == -1
        else:
            model += outgoing - incoming == 0

    model.solve(pulp.PULP_CBC_CMD(msg=False))
    status = pulp.LpStatus[model.status]

    if status != "Optimal":
        return {
            "status": status,
            "objective_value": None,
            "used_edges": [],
            "path": [],
        }

    objective_value = pulp.value(model.objective)
    used_edges = [
        edge
        for edge in problem.edges
        if pulp.value(edge_vars[(edge.from_node, edge.to_node)]) > 0.5
    ]

    # Search only selected edges and avoid revisiting nodes during reconstruction.
    selected_outgoing = {}
    for edge in used_edges:
        selected_outgoing.setdefault(edge.from_node, []).append(edge)

    def find_selected_path(current: str, visited: Set[str]):
        if current == problem.target:
            return []

        for edge in selected_outgoing.get(current, []):
            if edge.to_node in visited:
                continue
            remaining_path = find_selected_path(
                edge.to_node, visited | {edge.to_node}
            )
            if remaining_path is not None:
                return [edge] + remaining_path
        return None

    path_edges = find_selected_path(problem.source, {problem.source})
    if path_edges is None:
        raise RuntimeError("The selected edges do not contain a source-to-target path.")

    path = [problem.source]
    path.extend(edge.to_node for edge in path_edges)

    return {
        "status": status,
        "objective_value": objective_value,
        "used_edges": used_edges,
        "path": path,
    }


def solve_shortest_path_enumeration(problem: ShortestPathProblem) -> Dict:
    """Enumerate simple paths as a fallback for small networks."""

    if problem.source not in problem.nodes or problem.target not in problem.nodes:
        raise ValueError("The source and target must both be nodes in the network.")
    if problem.source == problem.target:
        raise ValueError("The source and target must be distinct nodes.")
    if any(edge.cost < 0 for edge in problem.edges):
        raise ValueError("This notebook assumes non-negative edge costs.")
    if any(
        edge.from_node not in problem.nodes or edge.to_node not in problem.nodes
        for edge in problem.edges
    ):
        raise ValueError("Every edge endpoint must be a node in the network.")
    edge_pairs = [(edge.from_node, edge.to_node) for edge in problem.edges]
    if len(edge_pairs) != len(set(edge_pairs)):
        raise ValueError("Each ordered node pair may have at most one directed edge.")

    all_paths = []

    def find_all_paths(current, visited, path):
        visited.add(current)
        path.append(current)

        if current == problem.target:
            all_paths.append(path.copy())
        else:
            for edge in problem.edges:
                if edge.from_node == current and edge.to_node not in visited:
                    find_all_paths(edge.to_node, visited, path)

        path.pop()
        visited.remove(current)

    find_all_paths(problem.source, set(), [])

    best_path = None
    best_cost = math.inf
    best_edges = []

    for path in all_paths:
        path_edges = []
        for start, end in zip(path, path[1:]):
            matching_edges = [
                edge
                for edge in problem.edges
                if edge.from_node == start and edge.to_node == end
            ]
            if not matching_edges:
                path_edges = []
                break
            path_edges.append(matching_edges[0])

        if path_edges or len(path) == 1:
            path_cost = sum(edge.cost for edge in path_edges)
            is_strictly_better = path_cost < best_cost and not math.isclose(
                path_cost, best_cost, rel_tol=1e-9, abs_tol=1e-9
            )
            if best_path is None or is_strictly_better:
                best_cost = path_cost
                best_path = path
                best_edges = path_edges

    if best_path is None:
        return {
            "status": "Infeasible",
            "objective_value": None,
            "path": [],
            "used_edges": [],
            "method": "Enumeration",
        }

    return {
        "status": "Optimal",
        "objective_value": best_cost,
        "path": best_path,
        "used_edges": best_edges,
        "method": "Enumeration",
    }


print("Shortest-path solver functions are defined.")

### Solving and checking the model

The solver returns selected edges, an objective value, a status, and a reconstructed route. The verification below calculates each node's net outflow directly from the selected edges rather than merely printing the expected balance. It also checks that the selected-edge costs sum to the reported objective. These checks connect the mathematical constraints to the computed result.

In [ ]:
solution = solve_shortest_path_mip(problem)
if solution["status"] != "Optimal":
    raise RuntimeError(f"The shortest-path model did not solve to optimality: {solution['status']}")

print("=== SHORTEST-PATH SOLUTION ===")
print(f"Status: {solution['status']}")
print(f"Minimum cost: {solution['objective_value']}")
print(f"Route: {' -> '.join(solution['path'])}")

print("\nSelected edges:")
for edge in solution["used_edges"]:
    print(f"  {edge.from_node} -> {edge.to_node}: cost = {edge.cost}")

selected_edge_cost = sum(edge.cost for edge in solution["used_edges"])
assert math.isclose(
    selected_edge_cost,
    solution["objective_value"],
    rel_tol=1e-9,
    abs_tol=1e-9,
), "The selected-edge cost must equal the reported objective value."

print("\nFlow-conservation check (net outflow = outgoing - incoming):")
for node_id in problem.nodes:
    outgoing_count = sum(
        1 for edge in solution["used_edges"] if edge.from_node == node_id
    )
    incoming_count = sum(
        1 for edge in solution["used_edges"] if edge.to_node == node_id
    )
    net_flow = outgoing_count - incoming_count

    if node_id == problem.source:
        expected_flow = 1
        role = "Source"
    elif node_id == problem.target:
        expected_flow = -1
        role = "Target"
    else:
        expected_flow = 0
        role = "Intermediate"

    assert net_flow == expected_flow, f"Flow conservation failed at node {node_id}."
    print(f"  {node_id} ({role}): net flow = {net_flow} ✓")

### Visual interpretation

The first panel shows every directed edge, with its edge cost as the label. The second panel emphasizes the selected edges and the route returned by the solver. Node coordinates are for visualization only; distances and coordinates do not alter the optimization model. The legend identifies the node roles and distinguishes selected links from other links.

In [ ]:
def visualize_network_and_solution(
    problem: ShortestPathProblem, solution: Dict
) -> None:
    """Plot the directed network and highlight the selected route."""

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    ax1.set_title("Directed freight network (edge labels: cost)", fontsize=13)

    for edge in problem.edges:
        from_node = problem.nodes[edge.from_node]
        to_node = problem.nodes[edge.to_node]
        ax1.annotate(
            "",
            xy=(to_node.x, to_node.y),
            xytext=(from_node.x, from_node.y),
            arrowprops={"arrowstyle": "->", "color": "gray", "lw": 1, "alpha": 0.6},
        )
        mid_x = (from_node.x + to_node.x) / 2
        mid_y = (from_node.y + to_node.y) / 2
        ax1.text(
            mid_x,
            mid_y,
            str(edge.cost),
            fontsize=8,
            ha="center",
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.8},
        )

    for node_id, node in problem.nodes.items():
        if node_id == problem.source:
            color = "red"
        elif node_id == problem.target:
            color = "blue"
        else:
            color = "lightgreen"
        ax1.scatter(node.x, node.y, s=300, c=color, edgecolors="black", zorder=5)
        ax1.text(node.x, node.y - 0.3, node_id, fontsize=12, ha="center", fontweight="bold")
        ax1.text(node.x, node.y - 0.5, node.name, fontsize=8, ha="center")

    ax1.set_xlabel("Horizontal plotting coordinate")
    ax1.set_ylabel("Vertical plotting coordinate")
    ax1.grid(True, alpha=0.3)
    ax1.set_aspect("equal")

    ax2.set_title("Selected shortest-path solution", fontsize=13)
    selected_pairs = {
        (edge.from_node, edge.to_node) for edge in solution.get("used_edges", [])
    }
    for edge in problem.edges:
        from_node = problem.nodes[edge.from_node]
        to_node = problem.nodes[edge.to_node]
        is_selected = (edge.from_node, edge.to_node) in selected_pairs
        ax2.annotate(
            "",
            xy=(to_node.x, to_node.y),
            xytext=(from_node.x, from_node.y),
            arrowprops={
                "arrowstyle": "->",
                "color": "red" if is_selected else "gray",
                "lw": 3 if is_selected else 1,
                "alpha": 0.85 if is_selected else 0.3,
            },
        )

    for node_id, node in problem.nodes.items():
        if node_id == problem.source:
            color = "red"
        elif node_id == problem.target:
            color = "blue"
        else:
            color = "lightgreen"
        size = 400 if node_id in solution["path"] else 300
        ax2.scatter(node.x, node.y, s=size, c=color, edgecolors="black", zorder=5)
        ax2.text(node.x, node.y - 0.3, node_id, fontsize=12, ha="center", fontweight="bold")

    ax2.set_xlabel("Horizontal plotting coordinate")
    ax2.set_ylabel("Vertical plotting coordinate")
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect("equal")
    ax2.text(
        0.5,
        -0.14,
        f"Route: {' -> '.join(solution['path'])}",
        transform=ax2.transAxes,
        fontsize=10,
        ha="center",
        bbox={"boxstyle": "round,pad=0.5", "facecolor": "white", "alpha": 0.8},
    )
    ax2.text(
        0.5,
        -0.23,
        f"Total cost: {solution['objective_value']}",
        transform=ax2.transAxes,
        fontsize=10,
        ha="center",
        bbox={"boxstyle": "round,pad=0.5", "facecolor": "white", "alpha": 0.8},
    )

    legend_handles = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="red",
               markeredgecolor="black", label="Source", markersize=9),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="blue",
               markeredgecolor="black", label="Target", markersize=9),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="lightgreen",
               markeredgecolor="black", label="Intermediate node", markersize=9),
        Line2D([0], [0], color="red", linewidth=3, label="Selected edge"),
        Line2D([0], [0], color="gray", linewidth=1, label="Other edge"),
    ]
    ax2.legend(handles=legend_handles, loc="upper left", fontsize=8)

    plt.tight_layout()
    plt.show()


visualize_network_and_solution(problem, solution)

### Sensitivity to edge-cost changes

Sensitivity analysis asks how the optimal route and its cost respond to a change in an input parameter. Here, the cost of each edge on the original optimal route is increased by 50% in a separate copy of the problem, and the same shortest-path solver is run for each copy. The original problem data are not modified. A route can change even when the minimum total cost does not increase, because a different route may become equally inexpensive or may already have the same minimum cost.

In [ ]:
def sensitivity_analysis(problem: ShortestPathProblem) -> List[Dict]:
    """Increase each selected edge cost by 50% and resolve the same model."""

    original_solution = solve_shortest_path_mip(problem)
    if original_solution["status"] != "Optimal":
        raise RuntimeError(
            f"Sensitivity analysis requires an optimal baseline solution; "
            f"received status {original_solution['status']}."
        )

    original_cost = original_solution["objective_value"]
    original_path = original_solution["path"]
    print("=== SENSITIVITY ANALYSIS ===")
    print(f"Original minimum cost: {original_cost}")
    print(f"Original route: {' -> '.join(original_path)}")

    sensitivity_results = []
    for selected_edge in original_solution["used_edges"]:
        modified_edges = []
        for edge in problem.edges:
            if (
                edge.from_node == selected_edge.from_node
                and edge.to_node == selected_edge.to_node
            ):
                modified_edges.append(
                    NetworkEdge(
                        edge.from_node,
                        edge.to_node,
                        edge.cost * 1.5,
                        edge.distance,
                    )
                )
            else:
                modified_edges.append(edge)

        modified_problem = ShortestPathProblem(
            problem.nodes, modified_edges, problem.source, problem.target
        )
        modified_solution = solve_shortest_path_mip(modified_problem)
        if modified_solution["status"] != "Optimal":
            raise RuntimeError(
                f"A sensitivity case did not solve to optimality: "
                f"{modified_solution['status']}."
            )

        path_changed = modified_solution["path"] != original_path
        sensitivity_results.append(
            {
                "edge": f"{selected_edge.from_node}->{selected_edge.to_node}",
                "original_cost": selected_edge.cost,
                "new_cost": selected_edge.cost * 1.5,
                "total_cost_increase": (
                    modified_solution["objective_value"] - original_cost
                ),
                "path_changed": path_changed,
                "new_path": " -> ".join(modified_solution["path"]),
            }
        )

    print("\nEffect of a 50% cost increase on each selected edge:")
    for result in sensitivity_results:
        print(
            f"Edge {result['edge']}: {result['original_cost']:.1f} "
            f"-> {result['new_cost']:.1f}"
        )
        print(f"  Change in minimum total cost: {result['total_cost_increase']:.1f}")
        print(f"  Route changed: {result['path_changed']}")
        if result["path_changed"]:
            print(f"  New route: {result['new_path']}")
        print()

    return sensitivity_results


sensitivity_results = sensitivity_analysis(problem)

### Checking a reference route

The route A → C → E → F → G is a feasible route in the supplied network and has cost $3+6+2+4=15$. Feasibility means that every consecutive pair is a directed edge in the network; it does not by itself establish optimality. The following check compares that reference route with the minimum cost found from the edge costs defined in this notebook. A reference route is optimal when its cost equals the computed minimum, even if the solver returns a different route with the same cost.

In [ ]:
def compare_with_expected_solution(
    problem: ShortestPathProblem, solution: Dict
) -> None:
    """Compare the computed optimum with the stated feasible reference route."""

    reference_edges = [("A", "C"), ("C", "E"), ("E", "F"), ("F", "G")]
    reference_path = ["A", "C", "E", "F", "G"]
    edge_costs = {
        (edge.from_node, edge.to_node): edge.cost for edge in problem.edges
    }

    missing_edges = [pair for pair in reference_edges if pair not in edge_costs]
    if missing_edges:
        raise ValueError(f"The reference route uses unavailable edges: {missing_edges}")

    reference_cost = sum(edge_costs[pair] for pair in reference_edges)
    route_matches = solution["path"] == reference_path
    cost_matches = math.isclose(
        solution["objective_value"], reference_cost, rel_tol=1e-9, abs_tol=1e-9
    )

    print("=== REFERENCE-ROUTE CHECK ===")
    print(f"Reference route: {' -> '.join(reference_path)}")
    print(f"Reference-route cost: {reference_cost}")
    print(f"Computed minimum-cost route: {' -> '.join(solution['path'])}")
    print(f"Computed minimum cost: {solution['objective_value']}")
    print(f"Computed route matches the reference route: {route_matches}")
    print(f"Computed cost matches the reference-route cost: {cost_matches}")
    print(
        "The reference route is optimal (its cost equals the computed minimum): "
        f"{cost_matches}"
    )


compare_with_expected_solution(problem, solution)

## Why this tier matters

The mathematical formulation makes the objective and the connectivity requirements explicit. It provides a precise model that can be checked, solved, and extended while preserving a clear interpretation of each decision variable and constraint. When an optimization solver reports an optimal status, its result is optimal for the stated model and input data.

### Strengths and limitations

**Strengths**
- Binary edge variables give a direct representation of which transportation links are selected.
- Flow-conservation constraints express the source, target, and intermediate-node requirements explicitly.
- The linear objective minimizes the total cost represented by the selected edges.
- The formulation provides a foundation to which additional, carefully specified constraints can be added.

**Limitations**
- A mathematical optimization solver is needed for the mixed-integer formulation used here.
- Writing and interpreting the constraints can be less immediate than tracing a route by hand on a small network.
- The formulation and this example do not include capacities, travel times, uncertainty, or other operational details.
- The included enumeration fallback is intended only for small networks because the number of simple paths can grow exponentially.

### When this formulation is useful

This approach is useful when the shortest-path problem needs an explicit mathematical model, when optimality matters, or when a network problem is to be developed further with additional constraints. The model's assumptions and objective must match the application: an optimum for edge cost is not necessarily an optimum for distance or travel time.

### Key takeaways

- Each directed edge has a binary selection variable.
- The objective minimizes the sum of selected edge costs.
- Flow conservation specifies the required net flow at the source, target, and intermediate nodes.
- The solver status, reconstructed route, selected-edge costs, and node balances together help interpret and verify a solution.
- The numerical result depends on the edge costs and other data supplied to the model.